# Phase 3B: Mobility Network & Temporal Intelligence

This notebook analyzes feed-scoped route-stop connectivity, shared-stop overlap, calendar weekday patterns, scheduled hourly profiles, service windows, and schedule concentration from the existing processed GTFS Parquet inputs. GTFS describes scheduled service; it does not establish passenger usage or demand. No forecasting or clustering is performed.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

project_root = Path.cwd()
if not (project_root / 'src').is_dir() and (project_root.parent / 'src').is_dir():
    project_root = project_root.parent
if not (project_root / 'src').is_dir():
    raise FileNotFoundError('Run from the repository root or notebooks directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.analysis.gtfs_eda import load_processed_tables
from src.analysis.gtfs_network import build_network_temporal_analysis

sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
input_dir = project_root / 'data' / 'processed' / 'gtfs'
gtfs_tables = load_processed_tables(input_dir)
analysis = build_network_temporal_analysis(gtfs_tables)

display(pd.DataFrame([{'table': name, 'rows': len(table), 'columns': list(table.columns)} for name, table in gtfs_tables.items()]))
display(analysis['bipartite_summary'])

## Methodology

The canonical `route_stops` table supplies one feed-scoped route-stop edge. Graph route nodes and stop nodes are separate typed partitions, and isolates remain represented. Route overlap is calculated only among pairs that share at least one stop; pairs not returned have zero Jaccard similarity. The heatmap below filters to the 20 routes with the most scheduled trip records in each feed.

Weekday trip counts use recurring `calendar` flags linked through `trips.service_id`. Hourly counts use scheduled stop departures; a blank departure falls back to arrival. Extended service-day hours are retained separately from wall-clock hour.

In [ ]:
degree = analysis['degree_distribution']
display(analysis['bipartite_summary'])
display(degree.groupby(['feed_id', 'node_type']).degree.describe().reset_index())

In [ ]:
feeds = sorted(degree.feed_id.unique())
fig, axes = plt.subplots(1, len(feeds), figsize=(7 * len(feeds), 4), squeeze=False)
for axis, feed_id in zip(axes[0], feeds):
    subset = degree.loc[degree.feed_id.eq(feed_id)]
    sns.histplot(data=subset, x='degree', hue='node_type', bins=40, ax=axis, element='step', stat='count')
    axis.set_yscale('log')
    axis.set(title=feed_id, xlabel='Bipartite node degree', ylabel='Node count (log scale)')
fig.suptitle('Route-stop bipartite degree distributions')
fig.tight_layout()
plt.show()

In [ ]:
stop_network = analysis['stop_connectivity']
top_connectivity = pd.concat([stop_network.loc[stop_network.feed_id.eq(feed)].nlargest(10, 'routes_per_stop') for feed in feeds], ignore_index=True)
top_scheduled_service = pd.concat([stop_network.loc[stop_network.feed_id.eq(feed)].nlargest(10, 'trips_per_stop') for feed in feeds], ignore_index=True)
display(top_connectivity)
display(top_scheduled_service)

In [ ]:
fig, axes = plt.subplots(1, len(feeds), figsize=(8 * len(feeds), 5), squeeze=False)
for axis, feed_id in zip(axes[0], feeds):
    subset = top_connectivity.loc[top_connectivity.feed_id.eq(feed_id)].copy()
    subset['label'] = subset['stop_name'].fillna(subset['stop_id']) + ' (' + subset['stop_id'] + ')'
    sns.barplot(data=subset, x='routes_per_stop', y='label', ax=axis, color='#3974a5')
    axis.set(title=feed_id, xlabel='Routes serving stop', ylabel='High-connectivity stop')
fig.suptitle('Top 10 high-connectivity stops per feed')
fig.tight_layout()
plt.show()

In [ ]:
overlap = analysis['route_overlap']
route_features = analysis['route_network_features']
display(overlap.groupby('feed_id').jaccard_similarity.agg(['count', 'mean', 'median', 'max']).reset_index())
display(overlap.sort_values(['jaccard_similarity', 'shared_stop_count'], ascending=[False, False]).groupby('feed_id').head(10))

In [ ]:
fig, axes = plt.subplots(1, len(feeds), figsize=(9 * len(feeds), 8), squeeze=False)
for axis, feed_id in zip(axes[0], feeds):
    selected = route_features.loc[route_features.feed_id.eq(feed_id)].nlargest(20, 'scheduled_trip_count').copy()
    selected = selected.sort_values('route_id')
    route_ids = selected.route_id.tolist()
    labels = (selected.route_short_name.fillna(selected.route_id) + ' | ' + selected.route_id).tolist()
    matrix = pd.DataFrame(0.0, index=route_ids, columns=route_ids)
    np.fill_diagonal(matrix.values, np.nan)
    local_pairs = overlap.loc[overlap.feed_id.eq(feed_id) & overlap.route_id_a.isin(route_ids) & overlap.route_id_b.isin(route_ids)]
    for pair in local_pairs.itertuples(index=False):
        matrix.loc[pair.route_id_a, pair.route_id_b] = pair.jaccard_similarity
        matrix.loc[pair.route_id_b, pair.route_id_a] = pair.jaccard_similarity
    sns.heatmap(matrix, mask=matrix.isna(), xticklabels=labels, yticklabels=labels, cmap='YlGnBu', vmin=0, vmax=1, cbar_kws={'label': 'Jaccard similarity'}, ax=axis)
    axis.set(title=f'{feed_id}: top 20 scheduled-trip routes (self-comparison masked)', xlabel='Route', ylabel='Route')
fig.suptitle('Route overlap by shared-stop Jaccard similarity')
fig.tight_layout()
plt.show()

## Temporal schedule profiles

Calendar weekdays are recurring schedule definitions, not observed operating days. The service-day hour axis below preserves values greater than 23; 25:30:00 is service-day hour 25, wall-clock hour 1 on offset 1. Neither profile establishes passenger peaks.

In [ ]:
display(analysis['weekday_profile'])
display(analysis['weekday_weekend_comparison'])
display(analysis['service_windows'].groupby('feed_id').agg(first_departure_seconds=('first_departure_seconds', 'min'), last_arrival_seconds=('last_arrival_seconds', 'max'), service_span_mean_seconds=('service_span_seconds', 'mean')).reset_index())

In [ ]:
weekday = analysis['weekday_profile']
fig, axis = plt.subplots(figsize=(10, 4))
sns.lineplot(data=weekday, x='weekday_order', y='scheduled_trip_count', hue='feed_id', marker='o', ax=axis)
axis.set_xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
axis.set(xlabel='Calendar weekday', ylabel='Scheduled trip records', title='Calendar weekday scheduled-service profile')
plt.tight_layout()
plt.show()

In [ ]:
hourly = analysis['hourly_profile']
fig, axes = plt.subplots(1, len(feeds), figsize=(8 * len(feeds), 4), squeeze=False)
for axis, feed_id in zip(axes[0], feeds):
    subset = hourly.loc[hourly.feed_id.eq(feed_id)]
    sns.lineplot(data=subset, x='service_day_hour', y='scheduled_departure_count', marker='o', ax=axis, color='#a3485a')
    axis.axvline(24, color='black', linestyle='--', linewidth=1)
    axis.set(title=feed_id, xlabel='Service-day hour (extended hours retained)', ylabel='Scheduled stop departures')
fig.suptitle('Hourly scheduled-service profile')
fig.tight_layout()
plt.show()

In [ ]:
display(analysis['service_concentration'])
curve = analysis['service_concentration_curve']
fig, axis = plt.subplots(figsize=(7, 5))
for feed_id in feeds:
    subset = curve.loc[curve.feed_id.eq(feed_id)]
    axis.plot(subset.route_rank_fraction, subset.cumulative_scheduled_trip_share, label=feed_id)
axis.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Equal schedule share reference')
axis.set(xlabel='Cumulative route share, ranked by scheduled trip records', ylabel='Cumulative scheduled-trip-record share', title='Scheduled-service concentration curve', xlim=(0, 1), ylim=(0, 1))
axis.legend()
plt.tight_layout()
plt.show()

In [ ]:
display(route_features.head(20))
display(route_features.groupby('feed_id').agg(routes=('route_id', 'nunique'), interval_routes=('scheduled_segment_count', 'count'), scheduled_trip_median=('scheduled_trip_count', 'median'), stops_per_route_median=('stops_per_route', 'median'), service_span_median_seconds=('service_span_seconds', 'median')).reset_index())

In [ ]:
features = route_features.copy()
features['log1p_scheduled_trips'] = np.log10(pd.to_numeric(features.scheduled_trip_count) + 1)
features['service_span_hours'] = pd.to_numeric(features.service_span_seconds, errors='coerce') / 3600
feature_specs = [
    ('log1p_scheduled_trips', 'log10(1 + scheduled trip records)'),
    ('stops_per_route', 'Distinct scheduled stops'),
    ('service_span_hours', 'Hours between first departure and last arrival'),
    ('mean_scheduled_stop_to_stop_minutes', 'Scheduled segment minutes'),
]
fig, axes = plt.subplots(len(feeds), len(feature_specs), figsize=(4.5 * len(feature_specs), 3.5 * len(feeds)), squeeze=False)
for row, feed_id in enumerate(feeds):
    subset = features.loc[features.feed_id.eq(feed_id)]
    for column, (feature, label) in enumerate(feature_specs):
        axis = axes[row, column]
        values = pd.to_numeric(subset[feature], errors='coerce').dropna()
        if values.empty:
            axis.text(0.5, 0.5, 'No derivable values', ha='center', va='center', transform=axis.transAxes)
        else:
            sns.histplot(x=values, bins=min(30, max(1, len(values))), ax=axis, color='#3974a5')
        axis.set(title=f'{feed_id} (n={len(subset)} routes)', xlabel=label, ylabel='Route count')
fig.suptitle('Route-network feature distributions, faceted by feed')
fig.tight_layout()
plt.show()

## Interpretation limits

- GTFS describes scheduled service, not actual operations, passenger usage, or passenger demand.
- Network degree and shared-stop overlap describe structural connectivity, not passenger use.
- Scheduled-service concentration and HHI describe trip records or route-stop edges, not passenger concentration.
- Calendar weekday patterns omit `calendar_dates` exceptions in these feeds.
- Extended service-day hours remain visible in `service_day_hour`; the separate `clock_hour` is not used to overwrite them.
- The overlap heatmap is limited to the top 20 routes by scheduled trip records within each feed. Stop charts show the top 10 stops within each feed.
- No clustering model, forecasting, or passenger-demand estimate is produced.